**Exercise 1**

Given the following data:

| Tid | Refund | Marital Status | Taxable Income (K) | Cheat |
|-----|--------|----------------|--------------------|-------|
| 1   | Yes    | Single         | 125                | No    |
| 2   | No     | Married        | 100                | No    |
| 3   | No     | Single         | 70                 | No    |
| 4   | Yes    | Married        | 120                | No    |
| 5   | No     | Divorced       | 95                 | Yes   |
| 6   | No     | Married        | 60                 | No    |
| 7   | Yes    | Divorced       | 220                | No    |
| 8   | No     | Single         | 85                 | Yes   |
| 9   | No     | Married        | 75                 | No    |
| 10  | No     | Single         | 90                 | Yes   |


What is the best first split, using Gini?

*Note, for the continuous feature, check quartile boundaries.*

In [5]:
# =============================================================================
# Decision Tree — Best First Split Using Gini Impurity
# =============================================================================
# Dataset: 10 records with features Refund, Marital Status, Taxable Income
# Target:  Cheat (Yes / No)
# Goal:    Find which feature/split minimises weighted Gini impurity
# =============================================================================

from itertools import combinations

# -----------------------------------------------------------------------------
# Dataset
# Each record is: (Tid, Refund, Marital Status, Taxable Income (K), Cheat)
# -----------------------------------------------------------------------------
data = [
    (1,  "Yes", "Single",   125, "No"),
    (2,  "No",  "Married",  100, "No"),
    (3,  "No",  "Single",    70, "No"),
    (4,  "Yes", "Married",  120, "No"),
    (5,  "No",  "Divorced",  95, "Yes"),
    (6,  "No",  "Married",   60, "No"),
    (7,  "Yes", "Divorced", 220, "No"),
    (8,  "No",  "Single",    85, "Yes"),
    (9,  "No",  "Married",   75, "No"),
    (10, "No",  "Single",    90, "Yes"),
]

# Extract just the class labels (Cheat column) for the full dataset
all_labels = [row[4] for row in data]


# =============================================================================
# Core Gini Function
# =============================================================================

def gini(labels):
    """
    Compute Gini impurity for a list of class labels.

    Gini = 1 - sum(p_i^2)  for each class i

    A perfectly pure node (all same class) has Gini = 0.
    Maximum impurity (equal split between 2 classes) has Gini = 0.5.

    Args:
        labels: list of class labels (e.g. ["Yes", "No", "No", ...])

    Returns:
        float: Gini impurity in [0, 0.5] for binary classification
    """
    n = len(labels)
    if n == 0:
        return 0.0  # empty node is perfectly pure by convention

    # Count occurrences of each class
    class_counts = {}
    for label in labels:
        class_counts[label] = class_counts.get(label, 0) + 1

    # Sum of squared probabilities
    sum_sq_probs = sum((count / n) ** 2 for count in class_counts.values())

    return 1.0 - sum_sq_probs


def weighted_gini(left_labels, right_labels):
    """
    Compute the weighted Gini impurity after a binary split.

    Weighted Gini = (|left| / |total|) * Gini(left)
                  + (|right| / |total|) * Gini(right)

    This is the quantity we want to MINIMISE — a lower value means
    the split produces purer child nodes.

    Args:
        left_labels:  class labels for the left child node
        right_labels: class labels for the right child node

    Returns:
        float: weighted Gini impurity of the split
    """
    n_total = len(left_labels) + len(right_labels)
    if n_total == 0:
        return 0.0

    w_left  = len(left_labels)  / n_total
    w_right = len(right_labels) / n_total

    return w_left * gini(left_labels) + w_right * gini(right_labels)


# =============================================================================
# Parent Node Gini (baseline)
# =============================================================================

parent_gini = gini(all_labels)

print("=" * 65)
print("GINI IMPURITY — BEST FIRST SPLIT ANALYSIS")
print("=" * 65)
print(f"\nDataset: {len(data)} records  |  "
      f"Yes (Cheat): {all_labels.count('Yes')}  |  "
      f"No (Cheat): {all_labels.count('No')}")
print(f"Parent Gini = 1 - (3/10)^2 - (7/10)^2 = {parent_gini:.3f}\n")


# =============================================================================
# Feature 1: Refund  (binary categorical: Yes / No)
# =============================================================================

print("-" * 65)
print("FEATURE 1: Refund")
print("-" * 65)

# Partition records by Refund value
refund_yes = [row[4] for row in data if row[1] == "Yes"]   # Tids 1, 4, 7
refund_no  = [row[4] for row in data if row[1] == "No"]    # Tids 2,3,5,6,8,9,10

gini_yes = gini(refund_yes)
gini_no  = gini(refund_no)
wg_refund = weighted_gini(refund_yes, refund_no)
gain_refund = parent_gini - wg_refund

print(f"  Refund=Yes  → {refund_yes}  Gini={gini_yes:.3f}")
print(f"  Refund=No   → {refund_no}")
print(f"             Gini={gini_no:.4f}")
print(f"\n  Weighted Gini = ({len(refund_yes)}/10)×{gini_yes:.3f}"
      f" + ({len(refund_no)}/10)×{gini_no:.4f} = {wg_refund:.3f}")
print(f"  Gini Gain     = {parent_gini:.3f} - {wg_refund:.3f} = {gain_refund:.3f}")


# =============================================================================
# Feature 2: Marital Status  (3-way categorical: Single / Married / Divorced)
#
# For a 3-class categorical feature we must evaluate all possible binary
# groupings. With 3 classes there are 3 non-trivial binary splits:
#   {Single} vs {Married, Divorced}
#   {Married} vs {Single, Divorced}
#   {Divorced} vs {Single, Married}
# =============================================================================

print("\n" + "-" * 65)
print("FEATURE 2: Marital Status")
print("-" * 65)

marital_categories = ["Single", "Married", "Divorced"]

# Group labels by marital status category
marital_groups = {
    cat: [row[4] for row in data if row[2] == cat]
    for cat in marital_categories
}

print("  Class distributions per category:")
for cat, labels in marital_groups.items():
    print(f"    {cat:10s} → {labels}  Gini={gini(labels):.3f}")

best_marital_wg   = float("inf")
best_marital_name = ""
best_marital_gain = 0.0

print("\n  Evaluating all binary groupings:")

# Generate all non-trivial binary partitions:
# For k categories, pick subsets of size 1..(k-1) for the left side.
# Using combinations of size 1 and 2 covers all unique splits without dupes.
for r in range(1, len(marital_categories)):  # r = size of left group
    for left_cats in combinations(marital_categories, r):
        right_cats = [c for c in marital_categories if c not in left_cats]

        # Avoid printing each split twice (left↔right are symmetric)
        # We print only when left_cats is "earlier" than right_cats
        if sorted(left_cats) > sorted(right_cats):
            continue

        left_labels  = [lbl for cat in left_cats  for lbl in marital_groups[cat]]
        right_labels = [lbl for cat in right_cats for lbl in marital_groups[cat]]

        wg   = weighted_gini(left_labels, right_labels)
        gain = parent_gini - wg

        left_str  = "{" + ", ".join(left_cats)  + "}"
        right_str = "{" + ", ".join(right_cats) + "}"
        print(f"    {left_str:20s} vs {right_str:25s} → "
              f"Weighted Gini={wg:.3f}  Gain={gain:.3f}")

        if wg < best_marital_wg:
            best_marital_wg   = wg
            best_marital_gain = gain
            best_marital_name = f"{left_str} vs {right_str}"

print(f"\n  Best Marital split: {best_marital_name}")
print(f"  Weighted Gini={best_marital_wg:.3f}  Gain={best_marital_gain:.3f}")


# =============================================================================
# Feature 3: Taxable Income  (continuous)
#
# For continuous features we find candidate thresholds at quartile boundaries:
#   Q1 ≈ 75,  Q2 (median) ≈ 92.5,  Q3 ≈ 110
#
# Each threshold t defines a binary split:  income ≤ t  vs  income > t
# =============================================================================

print("\n" + "-" * 65)
print("FEATURE 3: Taxable Income (continuous — quartile boundaries)")
print("-" * 65)

# Sorted income values: 60, 70, 75, 85, 90, 95, 100, 120, 125, 220
incomes = sorted(row[3] for row in data)
print(f"  Sorted incomes: {incomes}")

# Quartile midpoints between adjacent sorted values
# Q1 boundary: midpoint between 3rd and 4th values  → (75 + 85) / 2 = 80 … but
# the problem specifies checking Q1=75, Q2=92.5, Q3=110 explicitly.
quartile_thresholds = [75, 92.5, 110]
print(f"  Quartile thresholds checked: {quartile_thresholds}\n")

best_income_wg        = float("inf")
best_income_threshold = None
best_income_gain      = 0.0

for t in quartile_thresholds:
    left_labels  = [row[4] for row in data if row[3] <= t]
    right_labels = [row[4] for row in data if row[3] >  t]

    wg   = weighted_gini(left_labels, right_labels)
    gain = parent_gini - wg

    print(f"  Income ≤ {t:5.1f}K → left={left_labels}  Gini={gini(left_labels):.3f}")
    print(f"           Income > {t:5.1f}K → right={right_labels}  Gini={gini(right_labels):.3f}")
    print(f"           Weighted Gini={wg:.3f}  Gain={gain:.3f}\n")

    if wg < best_income_wg:
        best_income_wg        = wg
        best_income_threshold = t
        best_income_gain      = gain

print(f"  Best Income split: Income ≤ {best_income_threshold}K")
print(f"  Weighted Gini={best_income_wg:.3f}  Gain={best_income_gain:.3f}")


# =============================================================================
# Final Comparison — Pick the Overall Best Split
# =============================================================================

print("\n" + "=" * 65)
print("SUMMARY — ALL CANDIDATE SPLITS")
print("=" * 65)

candidates = [
    ("Refund (Yes vs No)",                    wg_refund,        gain_refund),
    (f"Marital Status ({best_marital_name})",  best_marital_wg,  best_marital_gain),
    (f"Income ≤ {best_income_threshold}K",     best_income_wg,   best_income_gain),
]

print(f"  {'Split':<40} {'Wtd Gini':>10}  {'Gain':>8}")
print(f"  {'-'*40} {'-'*10}  {'-'*8}")

best_split = min(candidates, key=lambda x: x[1])

for name, wg, gain in candidates:
    marker = "  ← BEST" if name == best_split[0] else ""
    print(f"  {name:<40} {wg:>10.3f}  {gain:>8.3f}{marker}")

print(f"\n✓ Best first split: {best_split[0]}")
print(f"  Weighted Gini = {best_split[1]:.3f}")
print(f"  Gini Gain     = {parent_gini:.3f} - {best_split[1]:.3f} = {best_split[2]:.3f}")
print("=" * 65)

GINI IMPURITY — BEST FIRST SPLIT ANALYSIS

Dataset: 10 records  |  Yes (Cheat): 3  |  No (Cheat): 7
Parent Gini = 1 - (3/10)^2 - (7/10)^2 = 0.420

-----------------------------------------------------------------
FEATURE 1: Refund
-----------------------------------------------------------------
  Refund=Yes  → ['No', 'No', 'No']  Gini=0.000
  Refund=No   → ['No', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes']
             Gini=0.4898

  Weighted Gini = (3/10)×0.000 + (7/10)×0.4898 = 0.343
  Gini Gain     = 0.420 - 0.343 = 0.077

-----------------------------------------------------------------
FEATURE 2: Marital Status
-----------------------------------------------------------------
  Class distributions per category:
    Single     → ['No', 'No', 'Yes', 'Yes']  Gini=0.500
    Married    → ['No', 'No', 'No', 'No']  Gini=0.000
    Divorced   → ['Yes', 'No']  Gini=0.500

  Evaluating all binary groupings:
    {Divorced}           vs {Single, Married}         → Weighted Gini=0.400  Gain=0.020
 

In [4]:
# =============================================================================
# Decision Tree — Full Tree Construction Using Gini Impurity
# =============================================================================
# Builds the complete tree from scratch, greedy top-down (CART-style):
#   At each node, try every possible split on every feature.
#   Pick the split with the lowest weighted Gini impurity.
#   Recurse on each child until a stopping condition is met.
# =============================================================================

from itertools import combinations

# -----------------------------------------------------------------------------
# Dataset
# -----------------------------------------------------------------------------
data = [
    (1,  "Yes", "Single",   125, "No"),
    (2,  "No",  "Married",  100, "No"),
    (3,  "No",  "Single",    70, "No"),
    (4,  "Yes", "Married",  120, "No"),
    (5,  "No",  "Divorced",  95, "Yes"),
    (6,  "No",  "Married",   60, "No"),
    (7,  "Yes", "Divorced", 220, "No"),
    (8,  "No",  "Single",    85, "Yes"),
    (9,  "No",  "Married",   75, "No"),
    (10, "No",  "Single",    90, "Yes"),
]

# Column indices within each tuple
IDX_REFUND  = 1
IDX_MARITAL = 2
IDX_INCOME  = 3
IDX_LABEL   = 4


# =============================================================================
# Gini Utilities  (same as before)
# =============================================================================

def gini(labels):
    """Gini impurity of a label list. Returns 0 for empty or pure nodes."""
    n = len(labels)
    if n == 0:
        return 0.0
    counts = {}
    for lbl in labels:
        counts[lbl] = counts.get(lbl, 0) + 1
    return 1.0 - sum((c / n) ** 2 for c in counts.values())


def weighted_gini(left_labels, right_labels):
    """Weighted Gini after a binary split."""
    n = len(left_labels) + len(right_labels)
    if n == 0:
        return 0.0
    return (len(left_labels) / n * gini(left_labels) +
            len(right_labels) / n * gini(right_labels))


def majority_class(labels):
    """Return the most common label (used for leaf prediction)."""
    counts = {}
    for lbl in labels:
        counts[lbl] = counts.get(lbl, 0) + 1
    return max(counts, key=counts.get)


# =============================================================================
# Split Candidate Generators
# =============================================================================

def refund_splits(records):
    """
    Generate the single binary split for the Refund feature (Yes / No).
    Yields: (description, left_records, right_records)
    """
    left  = [r for r in records if r[IDX_REFUND] == "Yes"]
    right = [r for r in records if r[IDX_REFUND] == "No"]
    if left and right:
        yield ("Refund = Yes", left, right)


def marital_splits(records):
    """
    Generate all non-trivial binary groupings for the 3-valued Marital Status
    feature.  For k categories there are (2^(k-1) - 1) unique binary splits.
    With k=3 that is 3 splits.
    Yields: (description, left_records, right_records)
    """
    categories = list({r[IDX_MARITAL] for r in records})
    if len(categories) < 2:
        return  # only one value present — no split possible

    for size in range(1, len(categories)):          # left-group sizes 1, 2, ...
        for left_cats in combinations(categories, size):
            right_cats = [c for c in categories if c not in left_cats]

            # Skip the mirror-image split to avoid duplicates
            if sorted(left_cats) > sorted(right_cats):
                continue

            left  = [r for r in records if r[IDX_MARITAL] in left_cats]
            right = [r for r in records if r[IDX_MARITAL] in right_cats]
            if left and right:
                desc = "{" + ",".join(sorted(left_cats)) + "}"
                yield (f"Marital in {desc}", left, right)


def income_splits(records):
    """
    Generate binary splits for the continuous Taxable Income feature.

    Candidate thresholds are the midpoints between every pair of adjacent
    distinct sorted values — the standard CART approach for continuous features.
    (The problem statement says 'check quartile boundaries', which corresponds
    to midpoints at Q1, median, Q3 of the sorted values.)

    Yields: (description, left_records, right_records)
    """
    sorted_vals = sorted({r[IDX_INCOME] for r in records})
    for i in range(len(sorted_vals) - 1):
        threshold = (sorted_vals[i] + sorted_vals[i + 1]) / 2
        left  = [r for r in records if r[IDX_INCOME] <= threshold]
        right = [r for r in records if r[IDX_INCOME] >  threshold]
        if left and right:
            yield (f"Income <= {threshold}K", left, right)


# =============================================================================
# Best Split Finder
# =============================================================================

def best_split(records):
    """
    Try every candidate split across all features.
    Return the split that minimises weighted Gini impurity.

    Returns a dict with keys:
        description  — human-readable split condition
        left         — records going left (condition True)
        right        — records going right (condition False)
        wgini        — weighted Gini of this split
        gain         — Gini gain vs the current node
    or None if no valid split exists.
    """
    labels        = [r[IDX_LABEL] for r in records]
    current_gini  = gini(labels)
    best          = None

    # Collect all candidate splits from every feature generator
    all_candidates = (
        list(refund_splits(records)) +
        list(marital_splits(records)) +
        list(income_splits(records))
    )

    for desc, left, right in all_candidates:
        left_labels  = [r[IDX_LABEL] for r in left]
        right_labels = [r[IDX_LABEL] for r in right]
        wg           = weighted_gini(left_labels, right_labels)
        gain         = current_gini - wg

        if best is None or wg < best["wgini"]:
            best = {
                "description": desc,
                "left":        left,
                "right":       right,
                "wgini":       wg,
                "gain":        gain,
            }

    return best


# =============================================================================
# Recursive Tree Builder
# =============================================================================

def build_tree(records, depth=0):
    """
    Recursively build a decision tree using greedy top-down splitting.

    Stopping conditions (a node becomes a leaf):
      1. All records have the same label  (pure node, Gini = 0)
      2. No valid split can be found      (e.g. only one record)

    Returns a nested dict representing the tree:
      Leaf node:     {"type": "leaf", "label": "Yes"/"No",
                      "records": [...], "gini": float}
      Internal node: {"type": "node", "split": str,
                      "left": <tree>, "right": <tree>,
                      "records": [...], "gini": float,
                      "wgini": float, "gain": float}
    """
    labels      = [r[IDX_LABEL] for r in records]
    node_gini   = gini(labels)
    tids        = [r[0] for r in records]

    indent = "  " * depth
    print(f"{indent}Node  tids={tids}  labels={labels}  gini={node_gini:.3f}")

    # --- Stopping condition 1: pure node ---
    if node_gini == 0.0:
        pred = labels[0]
        print(f"{indent}  → LEAF: {pred}  (pure)")
        return {"type": "leaf", "label": pred,
                "records": records, "gini": node_gini}

    # --- Find best split ---
    split = best_split(records)

    # --- Stopping condition 2: no valid split ---
    if split is None:
        pred = majority_class(labels)
        print(f"{indent}  → LEAF: {pred}  (no split, majority)")
        return {"type": "leaf", "label": pred,
                "records": records, "gini": node_gini}

    print(f"{indent}  Best split: [{split['description']}]"
          f"  wgini={split['wgini']:.3f}  gain={split['gain']:.3f}")

    # --- Recurse on children ---
    left_tree  = build_tree(split["left"],  depth + 1)
    right_tree = build_tree(split["right"], depth + 1)

    return {
        "type":    "node",
        "split":   split["description"],
        "left":    left_tree,           # records where condition is TRUE
        "right":   right_tree,          # records where condition is FALSE
        "records": records,
        "gini":    node_gini,
        "wgini":   split["wgini"],
        "gain":    split["gain"],
    }


# =============================================================================
# Pretty-Print the Tree
# =============================================================================

def print_tree(node, indent=0, branch_label="ROOT"):
    """
    Print the tree in a readable indented format.
    LEFT  = condition is TRUE
    RIGHT = condition is FALSE (the 'else' branch)
    """
    pad = "    " * indent

    if node["type"] == "leaf":
        tids = [r[0] for r in node["records"]]
        print(f"{pad}[{branch_label}]  → PREDICT: {node['label']}"
              f"  (tids={tids}, gini={node['gini']:.3f})")
    else:
        tids = [r[0] for r in node["records"]]
        print(f"{pad}[{branch_label}]  SPLIT on: {node['split']}"
              f"  (tids={tids}, gini={node['gini']:.3f},"
              f" gain={node['gain']:.3f})")
        # Left = condition TRUE, Right = condition FALSE
        cond_true  = node["split"]
        # Derive the FALSE label from the split description
        if "Marital in" in cond_true:
            cond_false = cond_true.replace("Marital in", "Marital NOT in")
        elif "<=" in cond_true:
            parts = cond_true.split("<=")
            cond_false = f"{parts[0].strip()} > {parts[1].strip()}"
        else:
            val = cond_true.split("=")[1].strip()
            feat = cond_true.split("=")[0].strip()
            other = "No" if val == "Yes" else "Yes"
            cond_false = f"{feat} = {other}"

        print_tree(node["left"],  indent + 1, f"TRUE:  {cond_true}")
        print_tree(node["right"], indent + 1, f"FALSE: {cond_false}")


# =============================================================================
# Predict Using the Tree
# =============================================================================

def predict(tree, record):
    """
    Traverse the tree for a single record and return the predicted label.
    The split description encodes which branch is 'left' (TRUE).
    """
    if tree["type"] == "leaf":
        return tree["label"]

    split = tree["split"]

    # Evaluate the split condition
    if split.startswith("Refund"):
        val = record[IDX_REFUND]
        goes_left = (val == "Yes")
    elif split.startswith("Marital"):
        # Extract the set of categories from "Marital in {A,B}"
        cats_str = split.split("{")[1].rstrip("}")
        cats = set(cats_str.split(","))
        goes_left = (record[IDX_MARITAL] in cats)
    elif "Income" in split:
        threshold = float(split.split("<=")[1].replace("K", "").strip())
        goes_left = (record[IDX_INCOME] <= threshold)
    else:
        raise ValueError(f"Unknown split format: {split}")

    return predict(tree["left"] if goes_left else tree["right"], record)


# =============================================================================
# Main
# =============================================================================

print("=" * 65)
print("BUILDING DECISION TREE")
print("=" * 65 + "\n")

tree = build_tree(data)

print("\n" + "=" * 65)
print("FINAL TREE STRUCTURE")
print("=" * 65 + "\n")
print_tree(tree)

print("\n" + "=" * 65)
print("PREDICTIONS ON TRAINING DATA")
print("=" * 65)
print(f"  {'Tid':>4}  {'Actual':>8}  {'Predicted':>10}  {'Correct':>8}")
print(f"  {'-'*4}  {'-'*8}  {'-'*10}  {'-'*8}")
correct = 0
for record in data:
    pred = predict(tree, record)
    actual = record[IDX_LABEL]
    ok = "✓" if pred == actual else "✗"
    if pred == actual:
        correct += 1
    print(f"  {record[0]:>4}  {actual:>8}  {pred:>10}  {ok:>8}")
print(f"\n  Accuracy: {correct}/{len(data)} = {correct/len(data)*100:.0f}%")
print("=" * 65)

Refund Gini: 0.3429
Marital Gini: 0.3

Quartile-based Income splits:
   Threshold  Left_n  Left_Yes  Left_No  Gini_Left  Right_n  Right_Yes  \
0       77.5       3         0        3     0.0000        7          3   
2      115.0       7         3        4     0.4898        3          0   
1       92.5       5         2        3     0.4800        5          1   

   Right_No  Gini_Right  Weighted_Gini  
0         4      0.4898         0.3429  
2         3      0.0000         0.3429  
1         4      0.3200         0.4000  


**Exercise 2**

Build a decision tree to fit the [federalist papers](https://www.kaggle.com/datasets/tobyanderson/federalist-papers) data. Note that you should restrict your analysis to papers written solely by Hamilton or Madison. Run your trained classifier on the "disputed" papers. What does your model tell you?

In [ ]:
# https://ist707.s3.us-east-2.amazonaws.com/data/federalist-papers.csv

**Exercise 3**

Build a **voting classifier** for the federalist papers, using all of the **non-ensemble** methods you've been exposed to in this class thus far (i.e., KNN, SVM, logistic regression, SGDClassifier, decision tree).

1) Compare this to a `RandomForest` classifier. Which works the best?

2) Compare this to a `GradientBoosting` classifier. Which works the best?

3) Add the `RandomForest` and `GradientBoosting` classifiers to your voting classifier. Does you performance improve?

**Exercise 4**

The "wine" dataset contains data about the chemical makeup of different varieties of wine and critics scores.  Use XGBoost to build a classifier for this data.  Manually tune the hyperparameters of the XGBoost model to try to achieve better accuracy on the test set than the baseline model. Some hyperparameters to consider tweaking:
   - `learning_rate`
   - `max_depth`
   - `n_estimators`
   - `gamma`
   - `subsample`
   - `colsample_bytree`

See [the online docs](https://xgboost.readthedocs.io/en/stable/parameter.html) for more info.

After tuning, use the `plot_importance` function again to see if feature importances have changed after tuning.


1. How did hyperparameter tuning affect the model's accuracy? Which hyperparameters seemed to have the most influence?
2. Did feature importances change after tuning? If so, why might that be?

In [ ]:
# Run this if you don't have XGBoost installed
%pip install XGBoost

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
import xgboost as xgb
from xgboost import plot_importance

data = load_wine()

# We'll use a data frame to make sure we get real feature names out
X = pd.DataFrame(data.data,columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = xgb.XGBClassifier(objective='multi:softprob', random_state=42)
clf.fit(X_train, y_train)

baseline_accuracy = clf.score(X_test, y_test)
print(f"Baseline Accuracy: {baseline_accuracy:.4f}")

plot_importance(clf)
plt.show()